# Zadanie dodatkowe, część 2 (Databricks Free Edition): MLflow, Unity Catalog i Model Serving

Notebook zakłada, że w Volume `workspace.default.zsn` leżą pliki z części 1: `cifar_cnn.keras` i `cifar_probka.npz`. Uruchamiacie go na serverless compute.

Kolejno:

1. instalujecie Keras 3 i wczytujecie model na backendzie PyTorch,
2. sprawdzacie, czy predykcje zgadzają się z tymi z Colaba,
3. opakowujecie model w `mlflow.pyfunc`, logujecie przebieg i rejestrujecie model w Unity Catalog,
4. wczytujecie zarejestrowaną wersję i sprawdzacie ją jeszcze raz,
5. tworzycie endpoint Model Serving (CPU, scale to zero), wysyłacie zapytanie i porównujecie odpowiedź,
6. usuwacie endpoint.

Uwaga: notebook nie był uruchamiany na Databricks. Części 1-4 sprawdziliśmy lokalnie na tych samych wersjach Keras i MLflow, część 5 przygotowaliśmy na podstawie dokumentacji. Jeśli coś nie działa, opiszcie to w raporcie, to też jest wynik.

## 0. Instalacja Keras 3

Środowisko serverless ma PyTorch, ale nie ma Keras ani TensorFlow. Po instalacji trzeba zrestartować proces Pythona, dlatego to osobna komórka.

In [ ]:
%pip install -q keras==3.13.2
dbutils.library.restartPython()

## 1. Konfiguracja

Zmienna `KERAS_BACKEND` musi być ustawiona przed pierwszym `import keras`.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import pandas as pd
import torch
import keras
import mlflow

print("Keras", keras.__version__, "backend", keras.backend.backend())
print("torch", torch.__version__, "| mlflow", mlflow.__version__)

KATALOG, SCHEMAT, WOLUMEN = "workspace", "default", "zsn"
KATALOG_PLIKOW = f"/Volumes/{KATALOG}/{SCHEMAT}/{WOLUMEN}"
NAZWA_MODELU = f"{KATALOG}.{SCHEMAT}.cifar_cnn"
NAZWA_ENDPOINTU = "zsn-cifar-cnn"   # nie może zaczynać się od "databricks-"

print(os.listdir(KATALOG_PLIKOW))

## 2. Wczytanie modelu i porównanie z Colabem

Model zapisany w TensorFlow wczytujemy na backendzie PyTorch. Różnice rzędu 1e-6 są normalne (inna kolejność operacji zmiennoprzecinkowych), ale klasy powinny się zgadzać.

In [ ]:
model = keras.saving.load_model(f"{KATALOG_PLIKOW}/cifar_cnn.keras")

dane = np.load(f"{KATALOG_PLIKOW}/cifar_probka.npz")
x, y, p_tf = dane["x"].astype("float32"), dane["y"], dane["p_tf"]
KLASY = [str(k) for k in dane["klasy"]]

p_torch = np.asarray(model.predict(x, verbose=0))
print("maksymalna różnica prawdopodobieństw:", float(np.abs(p_torch - p_tf).max()))
print("zgodność klas TF i torch:", float((p_torch.argmax(1) == p_tf.argmax(1)).mean()))
print("dokładność na próbce:", float((p_torch.argmax(1) == y).mean()))

## 3. Model pyfunc

`mlflow.pyfunc.PythonModel` to opakowanie, które MLflow i Model Serving umieją uruchomić bez wiedzy, czym jest model w środku. Plik `.keras` dołączamy jako artefakt. W `load_context` ustawiamy backend, bo endpoint uruchomi ten kod w osobnym kontenerze, w którym nikt wcześniej nie ustawił `KERAS_BACKEND`.

Model zwraca tabelę z numerem klasy, jej nazwą i pewnością.

In [ ]:
class CifarCNN(mlflow.pyfunc.PythonModel):
    KLASY = KLASY

    def load_context(self, context):
        import os
        os.environ["KERAS_BACKEND"] = "torch"
        import keras
        self.model = keras.saving.load_model(context.artifacts["keras_model"])

    def predict(self, context, model_input, params=None):
        import numpy as np
        import pandas as pd
        x = np.asarray(model_input, dtype="float32").reshape(-1, 32, 32, 3)
        p = np.asarray(self.model.predict_on_batch(x))
        k = p.argmax(axis=1)
        return pd.DataFrame({"klasa": k.astype("int64"),
                             "nazwa": [self.KLASY[i] for i in k],
                             "pewnosc": p.max(axis=1).astype("float64")})

## 4. Przebieg MLflow i rejestracja w Unity Catalog

Rejestr modeli w Unity Catalog wymaga sygnatury, czyli opisu wejścia i wyjścia. Tworzymy ją z przykładowych danych. Listę pakietów podajemy jawnie i przypinamy wersje z tego środowiska, żeby kontener endpointu miał te same biblioteki.

Podczas logowania MLflow wypisze kilka ostrzeżeń, między innymi o braku podpowiedzi typów w metodzie `predict`, o kolumnie całkowitoliczbowej w sygnaturze i o zapisie obiektu przez CloudPickle. W tym zadaniu nie trzeba nic z nimi robić.

In [ ]:
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

opakowany = CifarCNN()
class _Ctx:  # lokalny test opakowania przed logowaniem
    artifacts = {"keras_model": f"{KATALOG_PLIKOW}/cifar_cnn.keras"}
opakowany.load_context(_Ctx())
wynik = opakowany.predict(None, x[:5])
print(wynik)

sygnatura = infer_signature(x[:5], wynik)
wymagania = [
    f"keras=={keras.__version__}",
    f"torch=={torch.__version__.split('+')[0]}",
    f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}",
]
print(sygnatura)
print(wymagania)

In [ ]:
dokl = float((p_torch.argmax(1) == y).mean())

with mlflow.start_run(run_name="cifar_cnn_keras_torch") as run:
    mlflow.log_params({"keras_backend": "torch", "zrodlo_modelu": "Colab, TensorFlow",
                       "liczba_parametrow": model.count_params()})
    mlflow.log_metric("dokladnosc_probka_200", dokl)
    info = mlflow.pyfunc.log_model(
        name="model",
        python_model=CifarCNN(),
        artifacts={"keras_model": f"{KATALOG_PLIKOW}/cifar_cnn.keras"},
        signature=sygnatura,
        input_example=x[:2],
        pip_requirements=wymagania,
        registered_model_name=NAZWA_MODELU,
    )

wersja = info.registered_model_version
print("run:", run.info.run_id, "| model:", NAZWA_MODELU, "wersja", wersja)

## 5. Zarejestrowany model: wczytanie i alias

Wczytujemy model z rejestru tak, jak zrobiłby to ktoś inny w zespole, i nadajemy wersji alias `champion`. Ten krok to jednocześnie inferencja wsadowa bez endpointu. Jeśli w Waszym workspace endpoint nie zadziała, na tym etapie macie już działający, zarejestrowany model.

In [ ]:
from mlflow import MlflowClient

klient_mlflow = MlflowClient()
klient_mlflow.set_registered_model_alias(NAZWA_MODELU, "champion", wersja)

z_rejestru = mlflow.pyfunc.load_model(f"models:/{NAZWA_MODELU}@champion")
wynik_rej = z_rejestru.predict(x)
print("zgodność z predykcją w notebooku:", float((wynik_rej["klasa"].to_numpy() == p_torch.argmax(1)).mean()))
wynik_rej.head()

## 6. Endpoint Model Serving

W Free Edition dostępne są wyłącznie endpointy CPU i ich liczba jest ograniczona. `scale_to_zero_enabled` sprawia, że nieużywany endpoint nie zużywa zasobów. Budowa kontenera i uruchomienie endpointu trwa zwykle kilkanaście minut (dokumentacja podaje 10-25 minut dla zmiany konfiguracji). W tym czasie obejrzyjcie przebieg w zakładce Experiments i model w Catalog Explorer.

In [ ]:
import time
from mlflow.deployments import get_deploy_client

klient = get_deploy_client("databricks")

konfiguracja = {"served_entities": [{
    "entity_name": NAZWA_MODELU,
    "entity_version": str(wersja),
    "workload_size": "Small",
    "scale_to_zero_enabled": True,
}]}

try:
    klient.create_endpoint(name=NAZWA_ENDPOINTU, config=konfiguracja)
except Exception as e:
    print("create_endpoint:", e)
    print("Jeśli endpoint już istnieje, aktualizuję konfigurację.")
    klient.update_endpoint(endpoint=NAZWA_ENDPOINTU, config=konfiguracja)

In [ ]:
start = time.time()
while True:
    stan = klient.get_endpoint(NAZWA_ENDPOINTU)["state"]
    print(f"{(time.time() - start) / 60:5.1f} min  {stan}")
    if stan.get("ready") == "READY" and stan.get("config_update") == "NOT_UPDATING":
        break
    if stan.get("config_update") == "UPDATE_FAILED":
        raise RuntimeError("Budowa endpointu nie powiodła się, zobaczcie Serving > endpoint > Logs")
    time.sleep(60)

## 7. Zapytanie do endpointu

Endpoint przyjmuje JSON. Dla wejścia tensorowego format to `{"inputs": [...]}`, gdzie lista ma kształt `(n, 32, 32, 3)`. Pierwsze zapytanie po dłuższej przerwie może trwać dłużej, bo endpoint wstaje z zera.

In [ ]:
n = 16
odpowiedz = klient.predict(endpoint=NAZWA_ENDPOINTU, inputs={"inputs": x[:n].tolist()})
wynik_ep = pd.DataFrame(odpowiedz["predictions"])
wynik_ep["prawdziwa"] = [KLASY[i] for i in y[:n]]
print("zgodność z modelem w notebooku:", float((wynik_ep["klasa"].to_numpy() == p_torch[:n].argmax(1)).mean()))
wynik_ep

To samo zapytanie można wysłać z dowolnego miejsca przez REST. Poniżej przykład z `requests`. Adres i nagłówek uwierzytelnienia daje `WorkspaceClient` z Databricks SDK, który w notebooku konfiguruje się sam. Poza Databricks potrzebny byłby osobisty token dostępu w nagłówku `Authorization: Bearer ...`.

In [ ]:
import requests, json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
naglowki = w.config.authenticate()

t0 = time.time()
r = requests.post(f"{w.config.host}/serving-endpoints/{NAZWA_ENDPOINTU}/invocations",
                  headers=naglowki, json={"inputs": x[:4].tolist()}, timeout=120)
print(r.status_code, f"{(time.time() - t0) * 1000:.0f} ms")
print(json.dumps(r.json(), ensure_ascii=False, indent=1)[:800])

## 8. Sprzątanie

Usuńcie endpoint po zakończeniu. Liczba endpointów w Free Edition jest ograniczona, a dzienny limit zasobów jest wspólny dla całego konta. Model w rejestrze i przebieg MLflow możecie zostawić.

In [ ]:
klient.delete_endpoint(NAZWA_ENDPOINTU)
print("Endpoint usunięty.")

## 9. Wnioski

TODO: w 5-8 zdaniach odpowiedzcie na pytania z instrukcji (sekcja „Co oddajecie”).